# 📡 Project 3.3 — PWM Signal Decoder
**DSA for Mechatronics | Week 03 Project**

---

> **Submission:** Upload your completed `.ipynb` to BlackBoard before the deadline.  
> **Presentation:** You will run and explain your code live in class the following session.  
> **AI tools:** Allowed. You must understand and be able to explain every line you submit.

---

## 🎯 What you are building

A servo motor receives commands via **PWM (Pulse Width Modulation)** pulses.
The pulse width in microseconds (µs) encodes a movement command:

| Pulse width (µs) | Command   |
|---|---|
| < 1200           | `REVERSE` |
| 1200 – 1800      | `STOP`    |
| > 1800           | `FORWARD` |

You receive a raw list of 300 pulse widths recorded from a motor controller.
Buried somewhere in the signal is a known **calibration start sequence** of 5 specific pulses —
you need to find it.

Your pipeline:
1. **Decode** each pulse width to a command string  
2. **Count** how often each command appears  
3. **Find** the longest uninterrupted run of the same command  
4. **Locate** the hidden start sequence  

---

## 📐 Week 03 concepts you will practise

| Concept | Where used |
|---|---|
| **List comprehension** | Decoding all pulses in one line (Ex 1) |
| **Frequency counting (dict)** | Counting command occurrences (Ex 2) |
| **Two pointers / linear scan** | Longest run (Ex 3) |
| **Pattern matching** | Finding a sub-sequence (Ex 4) |
| **String formatting** | Timeline display (Ex 4) |

---

## 🔬 Background — Pattern Matching

Searching for a sequence inside a list is exactly like searching for a substring inside a string.
The naive (O(n×k)) approach:

```
signal  = [1000, 1500, 1000, 2000, 1500, 1000, 2000, 1000, 2000, 1000]
pattern = [1000, 2000, 1000]          ← k=3

For each starting position i in signal:
    if signal[i : i+k] == pattern:
        return i                       ← found at position i
```

This is O(n×k) because for each of the n positions we compare up to k elements.
For short patterns (k≤10) this is perfectly fast enough — no need for KMP or Rabin-Karp here.

---

## ⚙️ Step 0 — Generate PWM data

Run this cell first. Do **not** modify it.  
The start sequence is injected at a random position — it's your job to find it in Exercise 4.


In [ ]:
import random
random.seed(99)

# Known calibration start sequence (5 pulses)
START_SEQUENCE = [1000, 2000, 1000, 2000, 1000]

def _random_pulse():
    """Generate a random pulse weighted toward realistic motor use."""
    cmd = random.choices(['REVERSE', 'STOP', 'FORWARD'], weights=[1, 3, 2])[0]
    if cmd == 'REVERSE': return random.randint(900, 1199)
    if cmd == 'STOP':    return random.randint(1200, 1800)
    return random.randint(1801, 2100)

# Build 300 random pulses
pwm_signal = [_random_pulse() for _ in range(300)]

# Inject start sequence at a hidden position
inject_pos = random.randint(10, 280)
for i, val in enumerate(START_SEQUENCE):
    pwm_signal[inject_pos + i] = val

print(f"PWM signal: {len(pwm_signal)} pulses")
print(f"Range: {min(pwm_signal)} µs – {max(pwm_signal)} µs")
print(f"Start sequence {START_SEQUENCE} hidden somewhere in the signal...")
print()
print("First 15 pulses (µs):")
for i, v in enumerate(pwm_signal[:15]):
    print(f"  [{i:03d}] {v} µs")


---

## ✏️ Exercise 1 — Decode the signal

Write `decode_signal(pwm)` that converts a list of pulse widths into a list of command strings.

### Decoding rule:
```
pulse < 1200  → 'REVERSE'
1200 ≤ pulse ≤ 1800  → 'STOP'
pulse > 1800  → 'FORWARD'
```

### Challenge: do it in **one line** using a list comprehension with conditional expressions.

Python conditional expression syntax:
```python
value = 'A' if condition else 'B'
```

For three options (chain them):
```python
value = 'A' if c1 else ('B' if c2 else 'C')
```

### Expected:
```
['STOP', 'FORWARD', 'STOP', 'REVERSE', 'FORWARD', ...]   (300 elements)
```

> 💡 **Full list comprehension template:**
> ```python
> return ['REVERSE' if p < 1200 else ('FORWARD' if p > 1800 else 'STOP') for p in pwm]
> ```
> Understand this before copying it — the interviewer will ask you to explain it.


In [ ]:
def decode_signal(pwm):
    """
    Decode PWM pulse widths to command strings.
    
    Rules:  < 1200 µs → 'REVERSE'
            1200–1800 → 'STOP'
            > 1800 µs → 'FORWARD'
    
    Returns
    -------
    list of str — same length as pwm
    """
    # YOUR CODE: one-liner list comprehension
    pass


# ── Test ────────────────────────────────────────────────────────────────
commands = decode_signal(pwm_signal)

print(f"Decoded {len(commands)} commands")
print(f"Unique commands: {set(commands)}")
print()
print("First 15 decoded:")
for i in range(15):
    print(f"  [{i:03d}] {pwm_signal[i]:>5} µs  →  {commands[i]}")

# Quick sanity check
assert decode_signal([1000, 1500, 2000]) == ['REVERSE', 'STOP', 'FORWARD']
print("\n✅ Sanity check passed")


---

## ✏️ Exercise 2 — Count command frequencies

Write `count_commands(commands)` that returns a **dict** mapping each command to its count.
Then print a frequency table with percentages.

### What the function returns:
```python
{'FORWARD': 118, 'STOP': 151, 'REVERSE': 31}
```

### What you print:
```
Command frequencies (total: 300):
  FORWARD  :  118  ( 39.3%)  ████████████████████
  STOP     :  151  ( 50.3%)  █████████████████████████
  REVERSE  :   31  ( 10.3%)  █████
```

> 💡 **Building the frequency dict:**
> ```python
> freq = {}
> for cmd in commands:
>     if cmd in freq:
>         freq[cmd] += 1
>     else:
>         freq[cmd] = 1
> return freq
> ```
> Or the Pythonic one-liner using `.get()`:
> ```python
> freq[cmd] = freq.get(cmd, 0) + 1
> ```

> 💡 **Building the bar:**
> ```python
> bar_len = int(count / total * 30)   # scale to max 30 chars
> bar = '█' * bar_len
> ```


In [ ]:
def count_commands(commands):
    """
    Count how many times each command appears.
    
    Returns
    -------
    dict: {command_string: count}
    """
    freq = {}
    # YOUR CODE: iterate through commands, count each one
    pass
    return freq


# ── Test + print table ───────────────────────────────────────────────────
commands = decode_signal(pwm_signal)
freq     = count_commands(commands)
total    = len(commands)

print(f"Command frequencies (total: {total}):")
print()

# Sort by count descending for display
for cmd, count in sorted(freq.items(), key=lambda x: -x[1]):
    pct     = count / total * 100
    bar_len = int(count / total * 40)
    bar     = '█' * bar_len
    print(f"  {cmd:<10}: {count:>5}  ({pct:5.1f}%)  {bar}")

# Verify totals
assert sum(freq.values()) == total, "Counts don't add up!"
print("\n✅ Counts verified")


---

## ✏️ Exercise 3 — Longest command run

Write `longest_run(commands)` that finds the longest stretch of **consecutive identical commands**.

### Two-pointer approach:
```
Use one pointer to track the start of the current run,
advance through the list while commands[i] == commands[start].

current_start = 0
best = (commands[0], 0, 1)    # (command, start_index, length)

i = 1
while i < len(commands):
    if commands[i] == commands[current_start]:
        i += 1                    # extend current run
    else:
        current_start = i         # start new run
        i += 1
    # update best if current run length > best length
```

### Returns:
`(command_name, start_index, run_length)`

### Expected (example):
```
Longest run: STOP  starting at index 42  length 7
```

> 💡 **Tracking current run length:**
> ```python
> current_length = i - current_start
> if current_length > best[2]:
>     best = (commands[current_start], current_start, current_length)
> ```


In [ ]:
def longest_run(commands):
    """
    Find the longest consecutive run of the same command.
    
    Returns
    -------
    tuple: (command_string, start_index, run_length)
    """
    if not commands:
        return None
    
    best_cmd    = commands[0]
    best_start  = 0
    best_length = 1
    
    current_start = 0
    
    for i in range(1, len(commands)):
        if commands[i] != commands[current_start]:
            current_start = i           # start of new run
        
        current_length = i - current_start + 1
        
        # YOUR CODE: update best if current_length > best_length
        pass
    
    return (best_cmd, best_start, best_length)


# ── Test ────────────────────────────────────────────────────────────────
commands = decode_signal(pwm_signal)
cmd, start, length = longest_run(commands)

print(f"Longest run: {cmd}  starting at index {start}  length {length}")
print(f"Pulse values in that run: {pwm_signal[start : start+length]}")
print()

# Also find longest run for each command separately
print("Longest run per command:")
for target in ['FORWARD', 'STOP', 'REVERSE']:
    filtered = [(i, c) for i, c in enumerate(commands) if c == target]
    # Find longest consecutive block in original signal
    max_run = 0
    run = 0
    for i in range(len(commands)):
        if commands[i] == target:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    print(f"  {target:<10}: max {max_run} consecutive")


---

## ✏️ Exercise 4 — Find the hidden start sequence

Write `find_sequence(pwm, pattern)` that returns the **starting index** of the first occurrence
of `pattern` inside `pwm`, or `-1` if not found.

### Naive O(n×k) algorithm:
```
n = len(pwm)
k = len(pattern)

for i in range(n - k + 1):           # try each possible starting position
    if pwm[i : i+k] == pattern:      # slice and compare
        return i
return -1
```

> 💡 **Python list slicing makes this trivial:**
> ```python
> pwm[i : i+k] == pattern
> ```
> This compares two lists element by element and returns `True` if they are identical.
> You don't need to write a manual element-by-element comparison loop.

### Expected output:
```
✅ Start sequence found at index ???!
Pulses at that position: [1000, 2000, 1000, 2000, 1000]
```
(The `???` is hidden — that's what you're finding.)


In [ ]:
def find_sequence(pwm, pattern):
    """
    Find the first occurrence of pattern in pwm (exact match on pulse widths).
    
    Parameters
    ----------
    pwm     : list of int — the full PWM signal
    pattern : list of int — the sequence to search for
    
    Returns
    -------
    int — first index where pattern starts, or -1 if not found
    """
    n = len(pwm)
    k = len(pattern)
    
    for i in range(n - k + 1):
        # YOUR CODE: check if pwm[i:i+k] matches pattern
        pass
    
    return -1


# ── Test ────────────────────────────────────────────────────────────────
found = find_sequence(pwm_signal, START_SEQUENCE)

if found >= 0:
    print(f"✅ Start sequence found at index {found}!")
    print(f"   Pattern  : {START_SEQUENCE}")
    print(f"   Signal   : {pwm_signal[found : found + len(START_SEQUENCE)]}")
    print(f"   Commands : {decode_signal(pwm_signal[found : found + len(START_SEQUENCE)])}")
else:
    print("❌ Start sequence not found — check your code")

# Also try finding a pattern that doesn't exist
missing = [9999, 9999, 9999]
result = find_sequence(pwm_signal, missing)
print(f"\nSearching for {missing}: {'not found ✅' if result == -1 else 'found at ' + str(result)}")


---

## 🚀 Stretch Goal — ASCII Command Timeline

Print a compact visual timeline of the decoded commands, 50 pulses per row:

```
t=000-049: ▒▒▓▒▒░▒▒▒▓▒▒▒▒▒▓▓▒▒▒░▒▒▓▒▒▒▒▒▒▒▓▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
t=050-099: ▒▒▒▒▒▒▒▒▒▒▒▒▓▒▒▒▒▒▒▒▒▒░▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒
...

Legend: ▓=FORWARD  ▒=STOP  ░=REVERSE
```

```python
commands = decode_signal(pwm_signal)
char_map = {'FORWARD': '▓', 'STOP': '▒', 'REVERSE': '░'}
row_size = 50

print("Command Timeline")
print("Legend: ▓=FORWARD  ▒=STOP  ░=REVERSE")
print()
for row_start in range(0, len(commands), row_size):
    chunk = commands[row_start : row_start + row_size]
    row_str = ''.join(char_map[c] for c in chunk)
    print(f"t={row_start:03d}-{row_start+len(chunk)-1:03d}: {row_str}")
```
